# The Financial Resolution Agent
**Domain 1: Agentic Architecture & Orchestration**

A production-grade autonomous agent demonstrating:

| Concept | Implementation |
|---|---|
| MCP Tool Registry | 4 financial tools with Bedrock `toolConfig` |
| Deterministic Agentic Loop | `stop_reason`-driven, zero natural language parsing |
| PostToolUse Hook | Normalises Unix timestamps + status codes from `get_transactions` |
| Interception Hook | Blocks `process_refund` > $500 and forces `escalate_to_human` |

In [ ]:
!pip install -q anthropic rich

In [ ]:
import json
import time
from datetime import datetime, timezone
import anthropic
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich import print as rprint

console = Console()

# Uses ANTHROPIC_API_KEY from environment
client = anthropic.AnthropicBedrock(
    aws_region="ap-southeast-1",
)
MODEL = "apac.anthropic.claude-sonnet-4-6-20250514-v1:0"
REFUND_POLICY_LIMIT = 500.00

console.print(Panel(
    f"[green]✓[/green] Client ready\n[cyan]Model:[/cyan] {MODEL}\n[cyan]Refund limit:[/cyan] ${REFUND_POLICY_LIMIT:.2f}",
    title="Setup"
))

## Architecture Flow

```
User Dispute
     │
     ▼
┌─────────────────────────────────────────────────────┐
│              DETERMINISTIC AGENTIC LOOP              │
│                                                     │
│  send messages ──► Claude ──► inspect stop_reason   │
│       ▲                            │                │
│       │              ┌─────────────┴──────────────┐ │
│       │              │ tool_use    │    end_turn   │ │
│       │              ▼             ▼               │ │
│       │      execute tools      return final       │ │
│       │           │              response          │ │
│       │      ┌────┴──────┐                         │ │
│       │      │  HOOKS    │                         │ │
│       │      │           │                         │ │
│       │  PreToolUse  PostToolUse                   │ │
│       │  (intercept) (normalise)                   │ │
│       │      │           │                         │ │
│       └──────┴───────────┘                         │ │
│              append tool_result to history          │ │
└─────────────────────────────────────────────────────┘
```

## Section 1 — Tool Registry (MCP Integration)

Four tools simulate the fintech platform's backend APIs.
They intentionally return **raw/dirty data** (Unix timestamps, integer status codes)
to give the PostToolUse hook something to clean.

In [ ]:
# ── Simulated backend tool implementations ────────────────────────────────────

def get_customer_profile(customer_id: str) -> dict:
    """Returns account standing. Simulates a real CRM lookup."""
    profiles = {
        "C-1001": {"customer_id": "C-1001", "name": "Ana Reyes",     "status": 1, "verified": True,  "account_age_days": 730},
        "C-1002": {"customer_id": "C-1002", "name": "Marco Santos",  "status": 2, "verified": False, "account_age_days": 45},
        "C-1003": {"customer_id": "C-1003", "name": "Lei Chen",      "status": 1, "verified": True,  "account_age_days": 1200},
    }
    return profiles.get(customer_id, {"error": "Customer not found"})


def get_transactions(customer_id: str) -> dict:
    """
    Returns transactions with RAW legacy data:
    - 'ts' is Unix timestamp (not ISO 8601)
    - 'status_code' is integer (not human-readable)
    - 'amount_cents' is integer cents (not decimal dollars)
    The PostToolUse hook will normalise these before Claude sees them.
    """
    raw_data = {
        "C-1001": {
            "customer_id": "C-1001",
            "transactions": [
                {"txn_id": "T-9901", "ts": 1748736000, "amount_cents": 75000, "merchant": "ShopMart PH",    "status_code": 1},
                {"txn_id": "T-9902", "ts": 1748822400, "amount_cents": 32050, "merchant": "GrabFood",       "status_code": 3},
                {"txn_id": "T-9903", "ts": 1748908800, "amount_cents": 62000, "merchant": "Lazada",         "status_code": 2},
            ]
        },
        "C-1003": {
            "customer_id": "C-1003",
            "transactions": [
                {"txn_id": "T-8801", "ts": 1748649600, "amount_cents": 89900, "merchant": "Shopee SG",      "status_code": 3},
                {"txn_id": "T-8802", "ts": 1748736000, "amount_cents": 150000, "merchant": "Grab Transport", "status_code": 1},
            ]
        },
    }
    return raw_data.get(customer_id, {"error": "No transactions found"})


def process_refund(customer_id: str, txn_id: str, amount: float, reason: str) -> dict:
    """Executes a refund. The PreToolUse interception hook will block this if amount > $500."""
    return {
        "status": "refund_processed",
        "customer_id": customer_id,
        "txn_id": txn_id,
        "amount_refunded": amount,
        "reference": f"REF-{txn_id}-{int(time.time())}",
        "message": f"Refund of ${amount:.2f} issued successfully.",
    }


def escalate_to_human(customer_id: str, root_cause: str, amount: float, recommended_action: str) -> dict:
    """Routes the ticket to a human support queue with a structured handoff protocol."""
    return {
        "status": "escalated",
        "ticket_id": f"ESC-{customer_id}-{int(time.time())}",
        "customer_id": customer_id,
        "root_cause": root_cause,
        "amount": amount,
        "recommended_action": recommended_action,
        "message": "Ticket routed to Tier-2 support queue.",
    }


TOOL_REGISTRY = {
    "get_customer_profile": get_customer_profile,
    "get_transactions":     get_transactions,
    "process_refund":       process_refund,
    "escalate_to_human":    escalate_to_human,
}

console.print("[green]✓[/green] Tool registry ready (4 tools)")

In [ ]:
# ── Bedrock / Anthropic tool definitions ──────────────────────────────────────

TOOLS = [
    {
        "name": "get_customer_profile",
        "description": "Retrieve a customer's account standing, verification status, and account age.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string", "description": "The customer ID, e.g. C-1001"}
            },
            "required": ["customer_id"],
        },
    },
    {
        "name": "get_transactions",
        "description": "Fetch a customer's recent transactions including amounts, merchants, and statuses.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string", "description": "The customer ID"}
            },
            "required": ["customer_id"],
        },
    },
    {
        "name": "process_refund",
        "description": "Issue a refund to the customer's original payment method.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string"},
                "txn_id":      {"type": "string", "description": "Transaction ID to refund"},
                "amount":      {"type": "number", "description": "Refund amount in USD"},
                "reason":      {"type": "string", "description": "Reason for the refund"},
            },
            "required": ["customer_id", "txn_id", "amount", "reason"],
        },
    },
    {
        "name": "escalate_to_human",
        "description": "Escalate the case to a human agent with a structured handoff protocol.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id":         {"type": "string"},
                "root_cause":          {"type": "string", "description": "Analysis of why the dispute occurred"},
                "amount":              {"type": "number", "description": "Disputed amount in USD"},
                "recommended_action":  {"type": "string", "description": "What the human agent should do"},
            },
            "required": ["customer_id", "root_cause", "amount", "recommended_action"],
        },
    },
]

console.print("[green]✓[/green] Tool definitions (toolConfig schema) ready")

## Section 2 — Hooks

### Hook 1: PostToolUse — Data Normalisation
Intercepts the raw result of `get_transactions` **after** execution but **before** it is appended to conversation history.
Converts:
- `ts` (Unix timestamp) → ISO 8601 string
- `amount_cents` (integer) → `amount_usd` (float, 2 decimal places)
- `status_code` (int) → human-readable string

### Hook 2: PreToolUse — Policy Enforcement
Intercepts `process_refund` **before** execution.
Blocks if `amount > $500` and returns a structured policy error to the agent.

In [ ]:
STATUS_MAP = {
    1: "completed",
    2: "pending",
    3: "disputed",
    4: "failed",
    5: "reversed",
}


def post_tool_use_hook(tool_name: str, raw_result: dict) -> dict:
    """
    PostToolUse Hook — Data Normalisation.
    Only acts on get_transactions; passes all other tools through unchanged.
    """
    if tool_name != "get_transactions":
        return raw_result

    if "error" in raw_result:
        return raw_result

    normalised_txns = []
    for txn in raw_result.get("transactions", []):
        normalised_txns.append({
            "txn_id":     txn["txn_id"],
            "date":       datetime.fromtimestamp(txn["ts"], tz=timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
            "amount_usd": round(txn["amount_cents"] / 100, 2),
            "merchant":   txn["merchant"],
            "status":     STATUS_MAP.get(txn["status_code"], f"unknown({txn['status_code']})"),
        })

    normalised = dict(raw_result)
    normalised["transactions"] = normalised_txns
    normalised["_hook"] = "PostToolUse: timestamps→ISO8601, cents→USD, status_code→label"
    return normalised


def pre_tool_use_hook(tool_name: str, tool_input: dict) -> dict | None:
    """
    PreToolUse Hook — Policy Enforcement.
    Returns None to allow execution, or a dict error to BLOCK it.
    The error dict is returned directly to the agent as a tool_result.
    """
    if tool_name != "process_refund":
        return None  # allow

    amount = tool_input.get("amount", 0)
    if amount > REFUND_POLICY_LIMIT:
        console.print(
            f"  [bold red]⛔ HOOK BLOCKED:[/bold red] process_refund(amount=${amount:.2f}) "
            f"exceeds policy limit of ${REFUND_POLICY_LIMIT:.2f}"
        )
        return {
            "error": "policy_violation",
            "message": f"Amount ${amount:.2f} exceeds the automated refund policy limit of ${REFUND_POLICY_LIMIT:.2f}. "
                       "You must escalate this case to a human agent using the escalate_to_human tool. "
                       "Include the customer ID, root cause, refund amount, and recommended action.",
        }

    return None  # allow


console.print("[green]✓[/green] Hooks defined: post_tool_use_hook + pre_tool_use_hook")

## Section 3 — Deterministic Agentic Loop

Control flow is driven **entirely** by `stop_reason`.

```
while True:
    response = call Claude
    if stop_reason == "tool_use"  → execute tools (with hooks) → loop
    if stop_reason == "end_turn"  → return final text          → break
    else                          → raise (unexpected signal)
```

**Anti-patterns avoided:**
- ❌ `if "refund" in response.text` — fragile NLP parsing
- ❌ `for i in range(10)` — arbitrary iteration cap
- ✅ Only `stop_reason` determines what happens next

In [ ]:
SYSTEM_PROMPT = """
You are a financial resolution agent for a fintech platform.
Your job is to investigate customer disputes and resolve them by either issuing a refund or escalating to a human.

Investigation workflow:
1. Retrieve the customer profile to verify account standing.
2. Retrieve the customer's transactions to identify the disputed transaction.
3. If the refund amount is within policy, process the refund directly.
4. If the refund amount exceeds policy or a tool blocks you, escalate to a human with a full structured handoff.

Always complete your investigation before taking action.
Be concise and precise in your final response to the customer.
"""


def run_agent(user_message: str, max_retries: int = 5) -> str:
    """
    Deterministic agentic loop.
    Driven entirely by stop_reason — no natural language parsing.
    """
    messages = [{"role": "user", "content": user_message}]
    console.print(Panel(f"[bold]User:[/bold] {user_message}", border_style="white"))

    while True:
        # ── Call Claude with retry on throttling ──────────────────────────────
        for attempt in range(max_retries):
            try:
                response = client.messages.create(
                    model=MODEL,
                    max_tokens=2048,
                    system=SYSTEM_PROMPT,
                    tools=TOOLS,
                    messages=messages,
                )
                break
            except anthropic.RateLimitError:
                wait = 2 ** attempt
                console.print(f"  [red]⚠ Rate limited. Waiting {wait}s...[/red]")
                time.sleep(wait)
        else:
            raise RuntimeError("Exceeded max retries due to rate limiting.")

        stop_reason = response.stop_reason
        console.print(f"  [dim]stop_reason:[/dim] [yellow]{stop_reason}[/yellow]")

        # ── Termination condition ─────────────────────────────────────────────
        if stop_reason == "end_turn":
            final_text = next(b.text for b in response.content if b.type == "text")
            return final_text

        # ── Tool use handling ─────────────────────────────────────────────────
        if stop_reason == "tool_use":
            tool_results = []

            for block in response.content:
                if block.type != "tool_use":
                    continue

                tool_name  = block.name
                tool_input = block.input
                console.print(f"  [cyan]→ tool:[/cyan] [bold]{tool_name}[/bold]({tool_input})")

                # ── PreToolUse Hook (policy enforcement) ──────────────────────
                blocked = pre_tool_use_hook(tool_name, tool_input)
                if blocked is not None:
                    # Return the policy error directly to the agent — do NOT execute the tool
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(blocked),
                    })
                    continue

                # ── Execute the tool ──────────────────────────────────────────
                fn = TOOL_REGISTRY[tool_name]
                raw_result = fn(**tool_input)

                # ── PostToolUse Hook (data normalisation) ─────────────────────
                normalised_result = post_tool_use_hook(tool_name, raw_result)
                if normalised_result is not raw_result:
                    console.print(f"  [dim magenta]↳ PostToolUse hook normalised {tool_name} output[/dim magenta]")

                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(normalised_result),
                })

            # Append assistant turn + all tool results, then loop
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user",      "content": tool_results})
            continue

        raise RuntimeError(f"Unexpected stop_reason: {stop_reason}")


console.print("[green]✓[/green] run_agent() loop defined")

## Section 4 — Test Cases

### Test A — Small refund (under $500) → should auto-process
### Test B — Large refund (over $500) → hook blocks → agent escalates

In [ ]:
# ── Test A: Small refund — PreToolUse hook allows it ─────────────────────────
console.rule("[bold green]Test A — Small Refund (expected: auto-processed)[/bold green]")

result_a = run_agent(
    "Customer C-1001 is disputing transaction T-9902 from GrabFood. "
    "They were charged but the order was never delivered. Please investigate and resolve."
)

console.print(Panel(result_a, title="[green]Agent Response — Test A[/green]", border_style="green"))

In [ ]:
# ── Test B: Large refund — PreToolUse hook blocks → agent must escalate ───────
console.rule("[bold red]Test B — Large Refund > $500 (expected: escalated)[/bold red]")

result_b = run_agent(
    "Customer C-1003 is disputing transaction T-8802 from Grab Transport for $1,500. "
    "They claim they never made this transaction and suspect fraud. Please investigate and resolve."
)

console.print(Panel(result_b, title="[red]Agent Response — Test B[/red]", border_style="red"))

## Summary — What Each Test Demonstrates

| | Test A (small refund) | Test B (large refund) |
|---|---|---|
| **Tools called** | `get_customer_profile` → `get_transactions` → `process_refund` | `get_customer_profile` → `get_transactions` → `process_refund` (blocked) → `escalate_to_human` |
| **PostToolUse hook** | Normalises `get_transactions` timestamps + status codes | Same |
| **PreToolUse hook** | Allows `process_refund` (amount ≤ $500) | **Blocks** `process_refund` (amount > $500) |
| **Loop termination** | `stop_reason: end_turn` after refund confirmation | `stop_reason: end_turn` after escalation ticket created |
| **Anti-pattern avoided** | Loop driven by `stop_reason` only — zero NLP parsing | Same |

---

### Key Principle

> The agent cannot override the `$500` policy by reasoning its way around it.
> The hook operates **below** the model layer — it intercepts at the Python execution level,
> not at the prompt level. No system prompt instruction can bypass a programmatic gate.